# Chọn mô hình Fashion-MNIST cho demo

Chỉ train trên GPU Colab. Giữ nguyên mốc 5 epoch.

Tạo gói bằng `python 03_thuc_nghiem/scripts/pack_source.py`, rồi upload `03_thuc_nghiem/artifacts/source.zip` qua Files → Upload to session storage.

7 cấu hình; split 54k/6k/10k; tối đa 30 epoch; early stopping 6 epoch. Chọn checkpoint theo validation loss, cấu hình theo macro-F1 validation. Cấu hình thắng chạy 3 seed; test chỉ dùng sau khi chốt.

Bản đã chạy: https://colab.research.google.com/drive/1K9FNgM9mzUL55AmOaVcc_vIfLd5DKkP0

In [ ]:
import torch
from pathlib import Path
import zipfile

assert torch.cuda.is_available(), "Chọn GPU T4 trước khi chạy."
print("GPU:", torch.cuda.get_device_name(0))
PROJECT = Path("/content/fashion_selection")
PROJECT.mkdir(exist_ok=True)
with zipfile.ZipFile("/content/source.zip") as source:
    source.extractall(PROJECT)
STAGE = PROJECT / "03_thuc_nghiem"


## Huấn luyện và lựa chọn

Ô này chạy lại toàn bộ thí nghiệm. Nếu đã có artifact, chuyển đến phần xem kết quả.

In [ ]:
import subprocess
import sys

with subprocess.Popen(
    [sys.executable, "-u", "select_model.py"], cwd=STAGE,
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
) as process, open(PROJECT / "training.log", "w") as log:
    for line in process.stdout:
        print(line, end="", flush=True)
        log.write(line)
        log.flush()
    assert process.wait() == 0, "Xem lỗi ở cuối log."


## Kiểm chứng và phân tích

Kiểm tra kỹ thuật trên GPU, sau đó phân tích checkpoint đã chốt. Không dùng kết quả test để điều chỉnh mô hình.

In [ ]:
for script in ["verify_selection.py", "analyze_selection.py"]:
    with subprocess.Popen(
        [sys.executable, "-u", script], cwd=STAGE,
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
    ) as process:
        for line in process.stdout:
            print(line, end="", flush=True)
        assert process.wait() == 0, f"Kiểm tra lại {script}."


## Đọc kết quả

Validation dùng chọn cấu hình. Test dùng báo cáo cuối. Độ lệch chuẩn 3 seed mô tả mức dao động. Softmax cao vẫn có thể đoán sai.

In [ ]:
import json
import pandas as pd
from IPython.display import display, Image

RUN = Path((STAGE / "artifacts/latest_selection.txt").read_text())
print("Run:", RUN.name)
display(pd.read_csv(RUN / "validation_comparison.csv"))
display(pd.read_csv(RUN / "test_comparison.csv"))
print(json.loads((RUN / "stability.json").read_text()))
display(Image(filename=str(RUN / "analysis/confident_errors.png")))
display(Image(filename=str(RUN / "analysis/train_class_means.png")))


## Tải kết quả

Lưu ZIP vào `03_thuc_nghiem/artifacts/`. `demo_bundle/model.pt` là checkpoint đã chọn; giữ kèm cấu hình và preprocessing.

In [ ]:
from google.colab import files

RUN = Path((STAGE / "artifacts/latest_selection.txt").read_text())
files.download(str(RUN) + ".zip")
